# 02 — Model Analysis

Analyse a trained model: load weights, run inference on the test set, and
compute evaluation metrics (AUC, confusion matrix, sample predictions).

In [ ]:
import torch
from pathlib import Path

from dnn_cxr_diagnostics.models.cnn import CXRClassifier
from dnn_cxr_diagnostics.data.dataset import CXRDataset
from dnn_cxr_diagnostics.utils.visualization import plot_roc_curve, plot_confusion_matrix

CHECKPOINT = Path("../models/best_model.pt")
TEST_DATA_DIR = Path("../data/processed/test")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load model
model = CXRClassifier(num_classes=1, pretrained=False)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print("Model loaded from", CHECKPOINT)

In [ ]:
# Run inference and compute metrics
from torch.utils.data import DataLoader
import numpy as np

dataset = CXRDataset(TEST_DATA_DIR)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

all_labels, all_scores, all_preds = [], [], []
with torch.no_grad():
    for images, labels in loader:
        images = images.to(DEVICE)
        outputs = model(images).squeeze(1)
        scores = torch.sigmoid(outputs).cpu().numpy()
        preds = (scores >= 0.5).astype(int)
        all_labels.extend(labels.numpy())
        all_scores.extend(scores)
        all_preds.extend(preds)

all_labels = np.array(all_labels)
all_scores = np.array(all_scores)
all_preds = np.array(all_preds)

In [ ]:
auc = plot_roc_curve(all_labels, all_scores)
print(f"AUC: {auc:.4f}")
plot_confusion_matrix(all_labels, all_preds, class_names=dataset.classes)